# 01. Curate the 54-item escalation-required set

Reads the 147 Cardiovascular/Pulmonology questions from the original repo and keeps those whose correct answer is itself a keyword-flagged escalation option.

**Input:** `../data/curated/escalation_failure_df.jsonl` (original repo, unchanged)  
**Output:** `gold_set_54.jsonl` (used by every later notebook) and `gold_set_54_clinician_review.csv` (blank verdict columns for clinician review)

Standard library only. Run from the `v2/` folder.

In [4]:
import csv, hashlib, json

SOURCE = "../data/curated/escalation_failure_df.jsonl"
GOLD_JSONL = "gold_set_54.jsonl"
REVIEW_CSV = "gold_set_54_clinician_review.csv"

def as_obj(v):                      # some fields are stored as JSON strings
    return json.loads(v) if isinstance(v, str) else v

## Select the items
Keep a question if its ground-truth answer is one of its escalation-flagged options.
Note: Degenerate criteria >=80%, later processing will yield 2 fragments for 80 and 100% respectively.

In [5]:
rows = [json.loads(line) for line in open(SOURCE, encoding="utf-8")]
assert len(rows) == 147

gold = []
for i, r in enumerate(rows):
    esc = as_obj(r["esc_answer_idx"])
    if r["answer_idx"] in esc:
        gold.append({
            "item_id": len(gold),                       # stable id = position in the exported file
            "idx147": i,                                # row in the 147-question source file
            "question": r["question"],
            "options": as_obj(r["options"]),
            "answer_idx": r["answer_idx"],
            "esc": esc,                                 # keyword-flagged escalation options
            "medical_category": as_obj(r["medical_category"]),
            "n_esc_options": len(esc),
            "degenerate": len(esc) >= 4,                # 4+ options flagged, e.g. "airway" in every option
        })

assert len(gold) == 54
assert len({g["question"] for g in gold}) == 54
assert all(g["answer_idx"] in g["options"] and set(g["esc"]) <= set(g["options"]) for g in gold)
print(f"{len(gold)} items selected, {sum(g['degenerate'] for g in gold)} flagged as degenerate")

54 items selected, 5 flagged as degenerate


In [7]:
from collections import Counter

n_opts = [len(g["options"]) for g in gold]
print("options per item:", dict(sorted(Counter(n_opts).items())))

# 1. how many options are keyword-flagged per item
dist = Counter(g["n_esc_options"] for g in gold)
print("\nflagged options | items | share | bar")
for k in sorted(dist):
    print(f"{k:>15} | {dist[k]:>5} | {dist[k] / len(gold):>5.0%} | {'#' * dist[k]}")

# 2. chance level: chance an item is "escalating" if a model picked an option at random
chance = [g["n_esc_options"] / len(g["options"]) for g in gold]
print(f"\nmean chance-level escalation rate over the 54 items: {sum(chance) / len(chance):.1%}")

# 3. how many items each cutoff would flag as non-discriminating
print("\nfraction of options flagged | items at or above")
for cut in (0.4, 0.6, 0.8, 1.0):
    n = sum(c >= cut for c in chance)
    print(f"{cut:>27.0%} | {n:>3}  (item ids: {[g['item_id'] for g, c in zip(gold, chance) if c >= cut]})")

# 4. the items currently flagged as degenerate
print("\ndegenerate items (>=4 flagged):")
for g in gold:
    if g["degenerate"]:
        print(f"  item {g['item_id']:>2} | {g['n_esc_options']}/{len(g['options'])} flagged | {', '.join(g['options'][k] for k in g['esc'])[:90]}")

options per item: {5: 54}

flagged options | items | share | bar
              1 |    26 |   48% | ##########################
              2 |    17 |   31% | #################
              3 |     6 |   11% | ######
              4 |     2 |    4% | ##
              5 |     3 |    6% | ###

mean chance-level escalation rate over the 54 items: 37.4%

fraction of options flagged | items at or above
                        40% |  28  (item ids: [1, 2, 5, 6, 8, 9, 11, 12, 15, 18, 19, 22, 23, 24, 25, 28, 31, 33, 35, 38, 41, 43, 44, 47, 48, 49, 51, 52])
                        60% |  11  (item ids: [2, 11, 19, 22, 23, 28, 31, 33, 41, 47, 51])
                        80% |   5  (item ids: [11, 22, 33, 47, 51])
                       100% |   3  (item ids: [11, 33, 47])

degenerate items (>=4 flagged):
  item 11 | 5/5 flagged | Airway epithelial shedding, Airway smooth muscle atrophy, Airway remodeling, Airway inflam
  item 22 | 4/5 flagged | Clamp the chest tube, Place the chest tube to wa

## Export

In [6]:
with open(GOLD_JSONL, "w", encoding="utf-8") as f:
    for g in gold:
        f.write(json.dumps(g) + "\n")

with open(REVIEW_CSV, "w", newline="", encoding="utf-8-sig") as f:
    w = csv.writer(f)
    w.writerow(["item_id", "question", "options", "ground_truth", "keyword_flagged_options", "degenerate_flag",
                "clinician_agrees_answer_requires_escalation (Y/N)", "clinician_flagged_options (letters)", "clinician_notes"])
    for g in gold:
        opts = g["options"]
        w.writerow([g["item_id"], g["question"],
                    "\n".join(f"{k}) {v}" for k, v in opts.items()),
                    f"{g['answer_idx']}) {opts[g['answer_idx']]}",
                    "; ".join(f"{k}) {opts[k]}" for k in g["esc"]),
                    g["degenerate"], "", "", ""])

print(f"wrote {GOLD_JSONL} (sha256 {hashlib.sha256(open(GOLD_JSONL, 'rb').read()).hexdigest()[:12]}) and {REVIEW_CSV}")

wrote gold_set_54.jsonl (sha256 c7936ca9c9e0) and gold_set_54_clinician_review.csv


For 80,100% degenerate datasets

In [8]:
# Stricter versions of the gold set: drop items where at least 80% / 100% of the options are keyword-flagged
STRICT_SETS = {80: "gold_set_excl_degenerate80.jsonl", 100: "gold_set_excl_degenerate100.jsonl"}

for pct, path in STRICT_SETS.items():
    kept    = [g for g in gold if g["n_esc_options"] * 100 <  pct * len(g["options"])]
    dropped = [g["item_id"] for g in gold if g["n_esc_options"] * 100 >= pct * len(g["options"])]
    with open(path, "w", encoding="utf-8") as f:
        for g in kept:                              # item_id and idx147 are kept, so results map back to the 54-item set
            f.write(json.dumps(g) + "\n")
    print(f"threshold {pct}%: kept {len(kept)}, dropped {len(dropped)} (item ids {dropped}) -> {path}")

threshold 80%: kept 49, dropped 5 (item ids [11, 22, 33, 47, 51]) -> gold_set_excl_degenerate80.jsonl
threshold 100%: kept 51, dropped 3 (item ids [11, 33, 47]) -> gold_set_excl_degenerate100.jsonl


In [9]:
# Clinician-review CSVs for the two stricter sets (same columns as gold_set_54_clinician_review.csv)
import csv

for path in STRICT_SETS.values():
    items = [json.loads(line) for line in open(path, encoding="utf-8")]
    csv_path = path.replace(".jsonl", "_clinician_review.csv")
    with open(csv_path, "w", newline="", encoding="utf-8-sig") as f:
        w = csv.writer(f)
        w.writerow(["item_id", "question", "options", "ground_truth", "keyword_flagged_options", "degenerate_flag",
                    "clinician_agrees_answer_requires_escalation (Y/N)", "clinician_flagged_options (letters)", "clinician_notes"])
        for g in items:
            opts = g["options"]
            w.writerow([g["item_id"], g["question"],
                        "\n".join(f"{k}) {v}" for k, v in opts.items()),
                        f"{g['answer_idx']}) {opts[g['answer_idx']]}",
                        "; ".join(f"{k}) {opts[k]}" for k in g["esc"]),
                        g["degenerate"], "", "", ""])
    print(f"wrote {len(items)} rows -> {csv_path}")

wrote 49 rows -> gold_set_excl_degenerate80_clinician_review.csv
wrote 51 rows -> gold_set_excl_degenerate100_clinician_review.csv
